# Adatok betoltese

In [ ]:
# Adatok beolvasasa

import mne
import glob
import os

data_dir = "E:/Onlab/adatok/derivatives"

set_files = glob.glob(os.path.join(data_dir, "sub-*/eeg/*.set"))

all_raws = []
all_pids = []

for file_path in set_files:
    subj_id = os.path.basename(os.path.dirname(file_path))
    raw = mne.io.read_raw_eeglab(file_path, preload=True)
    
    all_raws.append(raw)
    all_pids.append(subj_id)

In [ ]:
# Cimkek beolvasasa

import pandas as pd

participants = pd.read_csv("E:/Onlab/adatok/participants.tsv", sep="\t")

labels = participants["Group"].tolist()

# Elofeldolgozas

In [ ]:
import numpy as np
import random

indices = list(range(len(all_raws)))
random.shuffle(indices)

split_indices = np.array_split(indices, 5)

x_chunks = [[all_raws[i] for i in idx_list] for idx_list in split_indices]
y_chunks = [[labels[i] for i in idx_list] for idx_list in split_indices]
pid_chunks = [[all_pids[i] for i in idx_list] for idx_list in split_indices]

print(f"Létrejött 5 vödör, vödrönként kb. {len(x_chunks[0])} alannyal.")

In [ ]:
def create_epochs(raw_list, labels, pid_list, epoch_length_sec=5):
    epoch_list = []
    epoch_labels = []
    epoch_pids = [] 
    
    for raw, label, pid in zip(raw_list, labels, pid_list):
        epochs = mne.make_fixed_length_epochs(raw, duration=epoch_length_sec, preload=True, verbose=False)
        
        epoch_list.append(epochs)
        epoch_labels.extend([label] * len(epochs))
        epoch_pids.extend([pid] * len(epochs))
    
    combined_epochs = mne.concatenate_epochs(epoch_list)
    return combined_epochs, epoch_labels, epoch_pids

In [ ]:
folded_x_data = []
folded_y_labels = []
folded_pids = []

freq = 256

for i in range(5):
    print(f"{i+1}. vödör epocholása...")
    
    epochs_out, labels_out, pids_out = create_epochs(
        x_chunks[i], 
        y_chunks[i], 
        pid_chunks[i]
    )

    epochs_out.resample(freq)
    
    folded_x_data.append(epochs_out)
    folded_y_labels.append(labels_out)
    folded_pids.append(pids_out)

In [ ]:
from sklearn.preprocessing import LabelEncoder

def scale_data(data, means=None, stds=None):
    if means is None or stds is None:
        means = np.mean(data, axis=(0, 2), keepdims=True)
        stds = np.std(data, axis=(0, 2), keepdims=True)
    
    scaled_data = (data - means) / (stds + 1e-6)
    return scaled_data, means, stds

le = LabelEncoder()
le.classes_ = np.array(['A', 'F', 'C'])

# Modell

In [ ]:
import itertools
import torch.nn as nn

selectedModel = "EEGMiner"

## EEGNet

In [ ]:
if selectedModel == "EEGNet":
    from braindecode.models import EEGNet

    epochs = 100
    
    param_grid = {
        'F1': [4, 8, 16],
        'D' : [1, 2, 4],
        'drop_prob': [0.25, 0.5],
        'activation': [nn.ReLU, nn.ELU],
        'lr': [0.001, 0.0005],
    }

    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]


def getModelEEGNet(params):
    return EEGNet(
        n_chans=params['n_chans'],
        n_outputs=params['n_outputs'],
        n_times=params['n_times'],

        kernel_length=64,
        F1=params['F1'],
        D=params['D'],
        F2 = params['F1'] * params['D'],
        drop_prob=params['drop_prob'],
        activation=params['activation']
    )

## EEGITNet

In [ ]:
if selectedModel == "EEGITNet":
    from braindecode.models import EEGITNet

    epochs = 100

    param_grid = {
        'n_filters_time': [19, 38], 
        'kernel_length': [16, 32],
        'tcn_kernel_size': [4, 8],
        'drop_prob': [0.3, 0.5],
        'lr': [0.001, 0.0005],
    }

    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

def getModelEEGITNet(params):
    calculated_tcn_in = params['n_filters_time'] * 7
    
    current_tcn_kernel = params['tcn_kernel_size']
    calculated_padding = current_tcn_kernel - 1
    
    return EEGITNet(
        n_chans=params['n_chans'],
        n_outputs=params['n_outputs'],
        n_times=params['n_times'],
        sfreq=params['sfreq'],
        n_filters_time=params['n_filters_time'],
        tcn_in_channel=calculated_tcn_in,
        kernel_length=params['kernel_length'],
        tcn_kernel_size=current_tcn_kernel,
        tcn_padding=calculated_padding,
        drop_prob=params['drop_prob']
    )


## EEGInception

In [ ]:
if selectedModel == "EEGInception":
    from braindecode.models import EEGInceptionERP

    n_channels = folded_x_data[0].shape[1]
    n_times = folded_x_data[0].shape[2]
    n_classes = len(le.classes_)
    sfreq = folded_x_data[0].info['sfreq']

    model = EEGInceptionERP(
        n_chans=n_channels,
        n_outputs=n_classes,
        n_times=n_times,
        sfreq=sfreq,
    )

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    criterion = nn.CrossEntropyLoss()

    print(f"A használt eszköz: {device}")

    epochs = 300

## EEGMiner


In [ ]:
if selectedModel == "EEGMiner":
    from braindecode.models import EEGMiner

    epochs = 150

    param_grid = {
        'method': ['mag', 'corr', 'plv'],
        'filter_f_mean': [[15.0, 15.0], [20.0, 20.0], [25.0, 25.0]],
        'filter_bandwidth': [[20.0, 20.0], [40.0, 40.0], [45.0, 45.0]],
        'lr': [0.001, 0.0005, 0.0001]
    }

    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]


def GetModelEEGMiner(params):
    return EEGMiner(
        n_chans=params['n_chans'],
        n_outputs=params['n_outputs'],
        n_times=params['n_times'],
        sfreq=params['sfreq'],
        method=params['method'],
        filter_f_mean=params['filter_f_mean'],
        filter_bandwidth=params['filter_bandwidth']
    )


# Tanitas

In [ ]:
import torch

n_epochs, n_channels, n_times = folded_x_data[0].get_data(copy=True).shape

base_config = {
    'n_chans' : n_channels,
    'n_times' : n_times,
    'n_outputs' : len(le.classes_),
    'sfreq' : folded_x_data[0].info['sfreq'],
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"A használt eszköz: {device}")

In [ ]:
from braindecode import EEGClassifier
from skorch.callbacks import LRScheduler, EarlyStopping, Checkpoint
from skorch.helper import predefined_split
from IPython.display import clear_output
from torch.utils.data import Dataset
import torch.optim as optim
import torch.nn as nn
import copy


class NumpyDataset(Dataset):
    def __init__(self, X, y, transform=False, noise_level=0.01):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
        self.transform = transform
        self.noise_level = noise_level

    def __len__(self): 
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        
        if self.transform:
            noise = torch.randn_like(x) * self.noise_level
            x = x + noise
            
        return x, y

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    
set_seed(42)

best_overall_acc = 0
total_combos = len(combinations)
p_idx = 1

for params in combinations:
    clear_output(wait=True)
    print(f"\n" + "="*50)
    print(f"ÖSSZESÍTETT HALADÁS: {p_idx} / {total_combos} kombináció")
    print(f"Konfiguráció: {params}")
    print(f"Előző rekord: {best_overall_acc:.2f}%")
    p_idx += 1

    current_config = {**base_config, **params}
    
    current_combo_best_acc = -1
    current_combo_best_state = None
    
    fold_accuracies = []
    all_fold_preds = []
    all_fold_true = []

    for test_fold_idx in range(5):
        print(f"--- FOLD {test_fold_idx + 1} INDÍTÁSA ---")

        if selectedModel == "EEGNet":
            model = getModelEEGNet(current_config)
        elif selectedModel == "EEGITNet":
            model = getModelEEGITNet(current_config)
        elif selectedModel == "EEGMiner":
            model = GetModelEEGMiner(current_config)
        
        
        val_fold_idx = (test_fold_idx + 1) % 5
        X_test_raw = folded_x_data[test_fold_idx].get_data()
        y_test_np = le.transform(folded_y_labels[test_fold_idx])
        
        X_val_raw = folded_x_data[val_fold_idx].get_data()
        y_val_np = le.transform(folded_y_labels[val_fold_idx])

        X_train_raw = np.concatenate([folded_x_data[j].get_data() for j in range(5) if j != test_fold_idx and j != val_fold_idx], axis=0)
        y_train_np = np.concatenate([le.transform(folded_y_labels[j]) for j in range(5) if j != test_fold_idx and j != val_fold_idx], axis=0)


        X_train_np, train_means, train_stds = scale_data(X_train_raw)
        X_val_np, _, _ = scale_data(X_val_raw, train_means, train_stds)
        X_test_np, _, _ = scale_data(X_test_raw, train_means, train_stds)


        valid_ds = NumpyDataset(X_val_np, y_val_np, transform=False)
        train_ds = NumpyDataset(X_train_np, y_train_np, transform=True, noise_level=0.01)


        clf = EEGClassifier(
            module=model,
            criterion=nn.CrossEntropyLoss,
            optimizer=optim.Adam,
            optimizer__lr=current_config['lr'] * 0.5,
            optimizer__weight_decay=1e-2,
            batch_size=512,
            max_epochs=epochs,
            train_split=predefined_split(valid_ds),
            device=device,
            iterator_train__shuffle=True,
            callbacks=[
                "accuracy",
                ("early_stopping", EarlyStopping(monitor='valid_loss', patience=35)),
                ("lr_scheduler", LRScheduler('CosineAnnealingLR', T_max=epochs)),
            ],
        )

        clf.fit(train_ds, y=None)

        y_pred = clf.predict(X_test_np)
        acc = 100 * np.mean(y_pred == y_test_np)
        
        if acc > current_combo_best_acc:
            current_combo_best_acc = acc
            current_combo_best_state = copy.deepcopy(clf.module_.state_dict())
        
        all_fold_preds.extend(y_pred)
        all_fold_true.extend(y_test_np)
        fold_accuracies.append(acc)
        
        print(f"Fold {test_fold_idx+1} kész. Teszt pontosság: {acc:.2f}%")

    avg_acc = np.mean(fold_accuracies)
    if avg_acc > best_overall_acc:
        best_overall_acc = avg_acc
        checkpoint = {
            'model_state_dict': current_combo_best_state,
            'params': current_config,
            'accuracy': avg_acc,
            'best_fold_acc': current_combo_best_acc,
            'fold_accuracies': fold_accuracies,
            'all_preds': all_fold_preds,
            'all_true': all_fold_true
        }
        torch.save(checkpoint, f'best_{selectedModel}_modell.pth')
        print(f">>> ÚJ REKORD! Átlag: {avg_acc:.2f}%")

    del model
    del clf
    torch.cuda.empty_cache()

# Kiertekeles

In [ ]:
import torch
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

data = torch.load(f'best_{selectedModel}_modell.pth')

print(f"Modell típusa: {data['params'].get('method', 'N/A')}")
print(f"Átlagos pontosság: {data['accuracy']:.2f}%")
print(f"Foldok pontosságai: {data['fold_accuracies']}")

cm = confusion_matrix(data['all_true'], data['all_preds'])
fig, ax = plt.subplots(figsize=(8, 6))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['A', 'F', 'C']) 
disp.plot(cmap='Blues', ax=ax)
plt.title(f"{selectedModel}\nA legjobb modell összesített teljesítménye: {data['accuracy']:.2f}%")
plt.show()